# PhishSense – XGBoost Model API

### Fixed Version
This notebook contains a fixed loading mechanism for the `XGBoostClassifier` file.

In [ ]:
import subprocess, sys
packages = ["flask", "flask-cors", "xgboost==1.7.6", "scikit-learn", "numpy", "pandas", "python-whois", "requests", "beautifulsoup4"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("All packages installed successfully.")

CalledProcessError: Command '['e:\\PhishSense Graduation Project\\Phish-Sense\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', 'xgboost==1.7.6', '-q']' returned non-zero exit status 1.

In [ ]:
import pickle, numpy as np, pandas as pd, ipaddress, re, requests as req, urllib, urllib.parse, urllib.request, whois, logging, os
from urllib.parse import urlparse
from datetime import datetime
from bs4 import BeautifulSoup
print("Libraries imported successfully.")

## Cell 3 – Robust Model Loading (FIXED)

In [ ]:
import xgboost as xgb
MODEL_PATH = r"E:\PhishSense Graduation Project\Phish-Sense\AI Model\XGBoostClassifier.pickle.dat"

model = None
try:
    with open(MODEL_PATH, "rb") as f:
        model = pickle.load(f)
    print(f"✅ Model loaded successfully via Pickle: {type(model).__name__}")
except Exception as e:
    print(f"⚠️ Standard pickle load failed: {e}")
    print("Attempting robust load for older XGBoost formats...")
    try:
        # Attempt loading as a dedicated XGBoost model instead of a generic pickle
        model = xgb.XGBClassifier()
        model.load_model(MODEL_PATH)
        print("✅ Model loaded successfully via XGBoost load_model!")
    except Exception as e2:
        print(f"❌ Robust load also failed: {e2}")

if model is not None:
    # Patch for older models missing metadata
    if not hasattr(model, 'n_features_in_'):
        model.n_features_in_ = 16
    print(f"Successfully initialized model with {model.n_features_in_} features.")
else:
    raise ValueError("Could not load model. Ensure the .dat file exists at: " + MODEL_PATH)

⚠️ Standard pickle load failed: [20:18:09] C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:989: Check failed: header[0] == '{' (
Attempting robust load for older XGBoost formats...
❌ Robust load also failed: sklearn needs to be installed in order to use this module


ValueError: Could not load model. Ensure the .dat file exists at: E:\PhishSense Graduation Project\Phish-Sense\AI Model\XGBoostClassifier.pickle.dat

In [ ]:
def getDomain(url):
    domain = urlparse(url).netloc
    if re.match(r"^www\.", domain): domain = re.sub(r"^www\.", "", domain)
    return domain

def havingIP(url):
    try: 
        ipaddress.ip_address(urlparse(url).netloc)
        return 1
    except: return 0

def haveAtSign(url): return 1 if "@" in url else 0
def getLength(url): return 0 if len(url) < 54 else 1
def getDepth(url):
    s = urlparse(url).path.split('/')
    return sum(1 for seg in s if len(seg) != 0)
def redirection(url):
    pos = url.rfind('//')
    return 1 if pos > 7 else 0
def httpDomain(url): return 1 if 'https' in urlparse(url).netloc else 0

shortening_services = r"bit\.ly|goo\.gl|shorte\.st|go2l\.ink|x\.co|ow\.ly|t\.co|tinyurl|tr\.im|is\.gd|cli\.gs|bit.do"

def tinyURL(url): return 1 if re.search(shortening_services, url) else 0
def prefixSuffix(url): return 1 if '-' in urlparse(url).netloc else 0

def web_traffic(url):
    try:
        response = req.get("http://localhost:5000/api/health", timeout=1) # Dummy check
        return 0 if response.status_code == 200 else 1
    except: return 1

def domainAge(domain_name):
    try:
        creation_date = domain_name.creation_date
        expiration_date = domain_name.expiration_date
        if isinstance(creation_date, list): creation_date = creation_date[0]
        if isinstance(expiration_date, list): expiration_date = expiration_date[0]
        age_days = abs((expiration_date - creation_date).days)
        return 1 if (age_days / 30) < 6 else 0
    except: return 1

def domainEnd(domain_name):
    try:
        expiration_date = domain_name.expiration_date
        if isinstance(expiration_date, list): expiration_date = expiration_date[0]
        today = datetime.now()
        end = abs((expiration_date.replace(tzinfo=None) - today).days)
        return 1 if (end / 30) < 6 else 0
    except: return 1

def iframe(html): return 1 if "<iframe" in html.lower() else 0
def mouseOver(html): return 1 if "onmouseover" in html.lower() else 0
def rightClick(html): return 1 if "event.button==2" in html.lower() else 0
def forwarding(html): return 1 if "window.location" in html.lower() else 0

In [ ]:
def extract_features(url):
    try: r = req.get(url, timeout=5, headers={'User-Agent': 'Mozilla/5.0'}); html = r.text
    except: html = ""
    
    dns = 0; domain_name = None
    try: domain_name = whois.whois(urlparse(url).netloc)
    except: dns = 1

    features = [
        havingIP(url), haveAtSign(url), getLength(url), getDepth(url), 
        redirection(url), httpDomain(url), tinyURL(url), prefixSuffix(url), 
        dns, web_traffic(url), 
        (1 if dns == 1 else domainAge(domain_name)), 
        (1 if dns == 1 else domainEnd(domain_name)), 
        iframe(html), mouseOver(html), rightClick(html), forwarding(html)
    ]
    return features

def predict(url):
    feat = extract_features(url)
    X = np.array(feat).reshape(1, -1)
    prediction = int(model.predict(X)[0])
    prob = model.predict_proba(X)[0]
    confidence = round(float(prob[prediction]) * 100, 2)
    return {"url": url, "label": "Phishing" if prediction == 1 else "Safe", "confidence": confidence, "riskScore": round(float(prob[1]) * 100, 2)}

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import threading

app = Flask(__name__); CORS(app)

@app.route('/predict', methods=['POST'])
def api_predict():
    url = request.json.get('url')
    return jsonify(predict(url))

@app.route('/health')
def health():
    return jsonify({"status": "ready", "model": str(type(model))})

def run(): app.run(host='0.0.0.0', port=5001, debug=False, use_reloader=False)
threading.Thread(target=run).start()
print("🚀 AI Server running on http://localhost:5001")